In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed42_c.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed2026_a_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single-1.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_seed123_d_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed3407_b_raw.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/best_single.csv
/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/submission_d4_seed42_c.csv
/kaggle/input

In [2]:
import pandas as pd
import numpy as np
 
COMP = '/kaggle/input/competitions/playground-series-s6e4/'
DS   = '/kaggle/input/datasets/mohankrishnathalla/predicting-irrigation-need-submission-dataset/'
 
sub = pd.read_csv(COMP + 'sample_submission.csv')
 
# Depth=3 raw (V16)
d3_a = pd.read_csv(DS + 'submission_seed2026_a_raw.csv').rename(columns={'Irrigation_Need':'d3a'})
d3_b = pd.read_csv(DS + 'submission_seed3407_b_raw.csv').rename(columns={'Irrigation_Need':'d3b'})
d3_c = pd.read_csv(DS + 'submission_seed42_c_raw.csv') .rename(columns={'Irrigation_Need':'d3c'})
d3_d = pd.read_csv(DS + 'submission_seed123_d_raw.csv').rename(columns={'Irrigation_Need':'d3d'})
 
# Depth=4 raw (V17)
d4_a = pd.read_csv(DS + 'submission_d4_seed2026_a_raw.csv').rename(columns={'Irrigation_Need':'d4a'})
d4_b = pd.read_csv(DS + 'submission_d4_seed3407_b_raw.csv').rename(columns={'Irrigation_Need':'d4b'})
d4_c = pd.read_csv(DS + 'submission_d4_seed42_c_raw.csv') .rename(columns={'Irrigation_Need':'d4c'})
d4_d = pd.read_csv(DS + 'submission_d4_seed123_d_raw.csv').rename(columns={'Irrigation_Need':'d4d'})
 
# BEST = d4 seed42 Optuna (LB 0.98005)
best = pd.read_csv(DS + 'submission_d4_seed42_c.csv').rename(columns={'Irrigation_Need':'BEST'})
 
dfs = (d3_a.merge(d3_b,on='id').merge(d3_c,on='id').merge(d3_d,on='id')
          .merge(d4_a,on='id').merge(d4_b,on='id').merge(d4_c,on='id').merge(d4_d,on='id')
          .merge(best,on='id'))
 
all8 = ['d3a','d3b','d3c','d3d','d4a','d4b','d4c','d4d']
dfs['all8_agree'] = dfs[all8].nunique(axis=1) == 1
 
print(f"All 8 agree: {dfs['all8_agree'].sum():,}")
print(f"Disagree:    {(~dfs['all8_agree']).sum():,}")
 
# Strategy: unanimous consensus, else best Optuna model
def vote(row):
    if row['all8_agree']:
        return row['d3a']   # any, they're all same
    return row['BEST']      # best Optuna on uncertain rows
 
sub['Irrigation_Need'] = dfs.apply(vote, axis=1)
sub.to_csv('submission_v4_optuna_fallback.csv', index=False)
print(f"Dist: {sub['Irrigation_Need'].value_counts().to_dict()}")
print("Saved submission_v4_optuna_fallback.csv")
 
# Also: depth=4 only — all4 agree else Optuna
d4_cols = ['d4a','d4b','d4c','d4d']
dfs['d4_agree'] = dfs[d4_cols].nunique(axis=1) == 1
print(f"\nD4-only agree: {dfs['d4_agree'].sum():,}")
 
sub2 = pd.read_csv(COMP + 'sample_submission.csv')
def vote_d4(row):
    if row['d4_agree']:
        return row['d4a']
    return row['BEST']
sub2['Irrigation_Need'] = dfs.apply(vote_d4, axis=1)
sub2.to_csv('submission_v4_d4vote_optuna.csv', index=False)
print(f"D4-vote dist: {sub2['Irrigation_Need'].value_counts().to_dict()}")
print("Saved submission_v4_d4vote_optuna.csv")

All 8 agree: 269,332
Disagree:    668
Dist: {'Low': 159508, 'Medium': 100525, 'High': 9967}
Saved submission_v4_optuna_fallback.csv

D4-only agree: 269,729
D4-vote dist: {'Low': 159512, 'Medium': 100784, 'High': 9704}
Saved submission_v4_d4vote_optuna.csv
